In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import numpy as np

In [2]:
from data import UncertainDataset, ds_single, ds_pressure

In [3]:
from pathlib import Path

In [4]:
def compute_min_max(key: str, data: xr.Dataset):
    min_ = np.nanmin(data[key].values) # ignore NANs in grid cells which are not classified
    max_ = np.nanmax(data[key].values)
    return min_, max_
    
    

In [5]:
def collect_info(key: str, data: xr.Dataset, grib_info: dict, level):
    parameterName = grib_info[key]["parameterName"]
    paramID = grib_info[key]["paramID"]
    parameterUnits = grib_info[key]["parameterUnits"]
    bitsPerValue = grib_info[key]["bitsPerValue"]
    min_, max_ = compute_min_max(key, data)
    error = (max_ - min_) / 2 ** bitsPerValue
    print(f"{parameterName}, {key}, {paramID}, {parameterUnits}, {level}, {min_}, {max_}, {error}")
    return (parameterName, key, paramID, parameterUnits, level, min_, max_, error)
    
    

In [6]:
from tqdm import tqdm
import eccodes

In [7]:
import re

def atoi(s: str):
    return int(s) if s.isdigit() else s

def natural_keys(s: str) -> list[str | int]:
    '''
    alist.sort(key=natural_keys) sorts in human order
    http://nedbatchelder.com/blog/200712/human_sorting.html
    '''
    return [atoi(c) for c in re.split(r"\d+", s)]

In [8]:
def get_grib_information(level):
    data_path = Path("/ec/res4/scratch/ecm5945/era5-data")
    grib = data_path / f"{level}_data" / "data_20240101.grib"
    grib_info = {}
    with eccodes.highlevel.FileReader(grib) as reader:
        for message in reader:

            parameterName = message.get("parameterName")
            shortName = message.get("shortName")
            paramID = message.get("paramId")
            if not paramID:
                paramID = message.get("ifsparam")
            parameterUnits = message.get("parameterUnits")
            bitsPerValue = message.get("bitsPerValue")
            dataType = message.get("dataType")

    
            line = f""
            if dataType == "em":
                grib_info[shortName] = {"parameterName": parameterName, "paramID": paramID, "parameterUnits": parameterUnits, "bitsPerValue": bitsPerValue}
    
    return grib_info

In [9]:
grib_info_pressure = get_grib_information("pressure")
grib_info_single = get_grib_information("single")

In [ ]:
with open("single_info.txt", "w") as file:
    file.write("Field Name\tShort Name\tgrib paramID\tUnit\tLevel\tMinimum value\tMaximum value\tExisting L1 error\n")
    for key in ds_single.mean:
        data = collect_info(key, ds_single.mean, grib_info_single, "Single/Surface")
        file.write("\t".join(map(str, data)) + "\n")

In [ ]:
with open("pressure_info.txt", "w") as file:
    file.write("Field Name\tShort Name\tgrib paramID\tUnit\tLevel\tMinimum value\tMaximum value\tExisting L1 error\n")
    for key in ds_pressure.mean:
        data = collect_info(key, ds_pressure.mean, grib_info_pressure, "Pressure")
        file.write("\t".join(map(str, data)) + "\n")


Potential vorticity, pv, 60, K m**2 kg**-1 s**-1, Pressure, -0.00028479727916419506, 0.00017763405048754066, 7.056142337091842e-09
Specific rain water content, crwc, 75, kg kg**-1, Pressure, 0.0, 0.0015198290348052979, 2.319075065315701e-08
Specific snow water content, cswc, 76, kg kg**-1, Pressure, 0.0, 0.004287123680114746, 6.541631591971964e-08
Geopotential, z, 129, m**2 s**-2, Pressure, -5969.50390625, 208893.0625, 3.2785425186157227
Temperature, t, 130, K, Pressure, 179.2017364501953, 325.6129455566406, 0.002234057756140828
